# 05 - Visualisations

Figures for the paper, written to `results/figures/`:

1. **Articles per day** by topic, per country, week-begin date ticks, campaign window shaded.
2. **Weekly diversity** (societal + political Dz) per country, campaign window shaded.
3. **Weekly diversity by topic** (immigration vs inflation), societal and political.
4. **Actor-group composition** per country.
5. **Actor-group composition** per country and topic.
6. **Representation per party** per country (coverage share vs parliamentary seat share).

The grey band marks the campaign period (final weeks before the dashed election line).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

import helpers as h

FIG_DIR = h.FIGURES_DIR
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

# --- data ---
soc = pd.read_csv(h.SOCIAL_CLEAN, sep=h.CSV_SEP)
soc["date"] = pd.to_datetime(soc["date"], format="mixed")
soc["topic_label"] = soc["topic"].map(h.TOPIC_LABEL).fillna(soc["topic"])
pol = pd.read_csv(h.POLITICAL_CLEAN, sep=h.CSV_SEP)
pol["topic_label"] = pol["topic"].map(h.TOPIC_LABEL).fillna(pol["topic"])

soc_cw  = pd.read_csv(h.RESULTS_DIR / "soc_div_country_week.csv")
pol_cw  = pd.read_csv(h.RESULTS_DIR / "pol_div_country_week.csv")
soc_cwt = pd.read_csv(h.RESULTS_DIR / "soc_div_country_week_topic.csv")
pol_cwt = pd.read_csv(h.RESULTS_DIR / "pol_div_country_week_topic.csv")
for d in (soc_cw, pol_cw, soc_cwt, pol_cwt):
    d["week_start"] = h.week_start(d["year_week"])

election = soc.groupby("country")["election_date"].first().map(pd.to_datetime)
campaign_start = soc_cw[soc_cw["campaign_period"] == 1].groupby("country")["week_start"].min()

def mark_campaign(ax, cc):
    """Shade the campaign window and mark its start (dotted) and election day (dashed)."""
    ax.axvspan(campaign_start[cc], election[cc], color="grey", alpha=0.10, zorder=0)
    ax.axvline(campaign_start[cc], color="grey", ls=":", lw=1)
    ax.axvline(election[cc], color="black", ls="--", lw=1)

# --- labels / colours ---
CAT_LABEL = {"nr_politicians": "Politicians", "nr_experts_professionals": "Experts/professionals",
             "nr_civil_society": "Civil society", "nr_general_public": "General public"}

# `topic_label` carries the short key used for filtering/grouping ("immigration",
# "inflation"); TOPIC_DISPLAY is the presentation string shown on figures. Keeping
# the two separate means we can change wording without touching any `== "inflation"`
# filter downstream (separation of data key from its label).
TOPIC_DISPLAY = {"immigration": "Immigration",
                 "inflation": "Rising Prices, Inflation and Cost of Living"}

# Okabe-Ito palette: an eight-colour qualitative scheme engineered to stay
# distinguishable under all common forms of colour-vision deficiency
# (deuteranopia, protanopia, tritanopia). Defining it once and referencing the
# entries by name below gives a single source of truth: a hue can be changed in
# one place and every figure follows (DRY).
OKABE_ITO = {"blue": "#0072B2", "orange": "#E69F00", "green": "#009E73",
             "vermillion": "#D55E00", "purple": "#CC79A7", "skyblue": "#56B4E9",
             "yellow": "#F0E442", "black": "#000000"}

# Each dict maps a *role within one figure* to a hue; within any single plot the
# hues are maximally separable. Reuse of a hue across figures is fine because each
# subplot carries its own legend, so the role is never ambiguous in context.

# Four societal actor groups (stacked in CAT_COLS order). Adjacent segments get
# alternating warm/cool hues so the stack stays readable: blue / orange / green / purple.
SOC_COLORS = {"Politicians": OKABE_ITO["blue"], "Experts/professionals": OKABE_ITO["orange"],
              "Civil society": OKABE_ITO["green"], "General public": OKABE_ITO["purple"]}

# Government vs opposition: vermillion vs sky blue, a warm/cool pair that is among
# the most reliably separable under colour-blindness and avoids the four societal hues
# (the two appear side by side in the composition figures).
POL_COLORS = {"Government": OKABE_ITO["vermillion"], "Opposition": OKABE_ITO["skyblue"]}

# Topic contrast (immigration vs inflation): same warm/cool split, kept consistent
# so immigration always reads "warm" and inflation "cool" across the line figures.
TOPIC_COLORS = {"immigration": OKABE_ITO["vermillion"], "inflation": OKABE_ITO["skyblue"]}

# The two series in the weekly-diversity figure. They are also distinguished by
# marker (o vs s), so the panel survives greyscale printing as well as colour-blindness.
SERIES_COLORS = {"societal": OKABE_ITO["blue"], "political": OKABE_ITO["vermillion"]}

print("loaded:", len(soc), "articles |", len(pol), "political rows")

## Composition helpers

In [ ]:
def soc_composition(df, keys):
    g = df.groupby(keys)[h.CAT_COLS].sum()
    g = g.div(g.sum(axis=1), axis=0)
    g.columns = [CAT_LABEL[c] for c in g.columns]
    return g

def pol_composition(df, keys):
    d = df.assign(Government=np.where(df["government"] == 1, df["nr_actors"].fillna(1), 0),
                  Opposition=np.where(df["government"] == 0, df["nr_actors"].fillna(1), 0))
    g = d.groupby(keys)[["Government", "Opposition"]].sum()
    return g.div(g.sum(axis=1), axis=0)

def stacked_bar(ax, comp, colors, title, ylabel="Share of actors"):
    x = np.arange(len(comp)); bottom = np.zeros(len(comp))
    for col in comp.columns:
        ax.bar(x, comp[col].values, bottom=bottom, label=col, color=colors[col], width=0.7)
        bottom += comp[col].values
    ax.set_xticks(x); ax.set_xticklabels([h.COUNTRY_NAME[c] for c in comp.index], rotation=25, ha="right")
    ax.set_ylim(0, 1); ax.set_ylabel(ylabel); ax.set_title(title); ax.grid(False)
print("helpers ready")

## 1. Articles per day (immigration vs inflation)

Week-begin date ticks; campaign window shaded, election dashed.

In [ ]:
fig, axes = plt.subplots(len(h.COUNTRY_ORDER), 1, figsize=(13, 16))
for ax, cc in zip(axes, h.COUNTRY_ORDER):
    sub = soc[soc["country"] == cc]
    daily = sub.groupby(["topic_label", sub["date"].dt.floor("D")]).size().rename("n").reset_index()
    for tp in ["immigration", "inflation"]:
        s = daily[daily["topic_label"] == tp]
        ax.plot(s["date"], s["n"], color=TOPIC_COLORS[tp], lw=1.1, label=TOPIC_DISPLAY[tp])
    mark_campaign(ax, cc)
    ax.set_title(h.COUNTRY_NAME[cc]); ax.set_ylabel("articles/day")
    ax.set_xlabel("date"); ax.set_xlim(sub["date"].min(), sub["date"].max())
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b %Y"))
    plt.setp(ax.get_xticklabels(), rotation=90, fontsize=6.5)
    ax.grid(True, axis="y", alpha=0.3)
axes[0].legend(title="topic", loc="upper left")
fig.suptitle("Articles per day by topic (campaign window shaded)", y=0.995, fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1_articles_per_day.png", bbox_inches="tight")
plt.show()

## 2. Weekly diversity (societal and political)

In [ ]:
fig, axes = plt.subplots(len(h.COUNTRY_ORDER), 1, figsize=(13, 15), sharey=True)
for ax, cc in zip(axes, h.COUNTRY_ORDER):
    s = soc_cw[soc_cw["country"] == cc].sort_values("week_start")
    p = pol_cw[pol_cw["country"] == cc].sort_values("week_start")
    ax.plot(s["week_start"], s["simpson_Dz"], color=SERIES_COLORS["societal"], lw=1.6, marker="o", ms=3, label="societal")
    ax.plot(p["week_start"], p["simpson_Dz"], color=SERIES_COLORS["political"], lw=1.6, marker="s", ms=3, label="political")
    mark_campaign(ax, cc)
    ax.set_title(h.COUNTRY_NAME[cc]); ax.set_ylabel("Simpson's Dz"); ax.set_ylim(0, 1)
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
axes[0].legend(loc="lower left", ncol=2)
fig.suptitle("Weekly societal and political actor diversity", y=0.995, fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_weekly_diversity.png", bbox_inches="tight")
plt.show()

## 3. Weekly diversity by topic

Same weekly Dz, split into immigration vs inflation. Left column societal, right column political.

In [ ]:
fig, axes = plt.subplots(len(h.COUNTRY_ORDER), 2, figsize=(14, 16), sharey=True)
for i, cc in enumerate(h.COUNTRY_ORDER):
    for j, (name, df) in enumerate([("Societal", soc_cwt), ("Political", pol_cwt)]):
        ax = axes[i, j]
        sub = df[df["country"] == cc]
        for tp in ["immigration", "inflation"]:
            s = sub[sub["topic_label"] == tp].sort_values("week_start")
            ax.plot(s["week_start"], s["simpson_Dz"], color=TOPIC_COLORS[tp], lw=1.3, marker="o", ms=2.5, label=TOPIC_DISPLAY[tp])
        mark_campaign(ax, cc)
        ax.set_ylim(0, 1)
        if i == 0: ax.set_title(name)
        if j == 0: ax.set_ylabel(f"{h.COUNTRY_NAME[cc]}\nSimpson's Dz", fontsize=9)
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=7)
axes[0, 0].legend(loc="lower left", fontsize=8, title="topic")
fig.suptitle("Weekly diversity by topic (immigration vs rising prices/inflation)", y=0.997, fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2b_weekly_diversity_by_topic.png", bbox_inches="tight")
plt.show()

## 4. Actor-group composition per country

In [ ]:
soc_c = soc_composition(soc, "country").reindex(h.COUNTRY_ORDER)
pol_c = pol_composition(pol, "country").reindex(h.COUNTRY_ORDER)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
stacked_bar(ax1, soc_c, SOC_COLORS, "Societal actor groups")
stacked_bar(ax2, pol_c, POL_COLORS, "Political actor groups (gov vs opp)")
ax1.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=4, frameon=False)
ax2.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=False)
fig.suptitle("Actor-group composition per country", y=1.0, fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3_composition_country.png", bbox_inches="tight")
plt.show()

## 5. Actor-group composition per country and topic

In [ ]:
soc_ct = soc_composition(soc, ["country", "topic_label"])
pol_ct = pol_composition(pol, ["country", "topic_label"])
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for j, tp in enumerate(["immigration", "inflation"]):
    stacked_bar(axes[0, j], soc_ct.xs(tp, level="topic_label").reindex(h.COUNTRY_ORDER), SOC_COLORS, f"Societal - {TOPIC_DISPLAY[tp]}")
    stacked_bar(axes[1, j], pol_ct.xs(tp, level="topic_label").reindex(h.COUNTRY_ORDER), POL_COLORS, f"Political - {TOPIC_DISPLAY[tp]}")
axes[0, 1].legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
axes[1, 1].legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
fig.suptitle("Distribution of Societal and Political Actors per Country and Topic", y=1.0, fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig4_composition_country_topic.png", bbox_inches="tight")
plt.show()

## 6. Representation per party per country

Per country, each party's share of political-actor coverage (bar, coloured by government /
opposition), with its parliamentary seat share marked (diamond). Bars longer than the diamond =
over-represented relative to seats (the reflective benchmark behind H2).

In [ ]:
party = (pol.groupby(["country", "party_final"])
            .agg(actors=("nr_actors", "sum"), gov=("government", "first"),
                 seats=("seats", "first")).reset_index())
party["cov"] = party.groupby("country")["actors"].transform(lambda s: s / s.sum())

counts = party.groupby("country").size().reindex(h.COUNTRY_ORDER)
fig, axes = plt.subplots(len(h.COUNTRY_ORDER), 1, figsize=(10, 0.42 * counts.sum() + 2),
                         gridspec_kw={"height_ratios": counts.values})
for ax, cc in zip(axes, h.COUNTRY_ORDER):
    d = party[party["country"] == cc].sort_values("cov")
    y = np.arange(len(d))
    colors = [POL_COLORS["Government"] if g == 1 else POL_COLORS["Opposition"] for g in d["gov"]]
    ax.barh(y, d["cov"], color=colors, height=0.7, zorder=2)
    ax.scatter(d["seats"], y, color="black", marker="D", s=20, zorder=3)
    ax.set_yticks(y); ax.set_yticklabels(d["party_final"], fontsize=8)
    ax.set_xlim(0, max(d["cov"].max(), d["seats"].max()) * 1.08)
    ax.set_title(h.COUNTRY_NAME[cc], fontsize=11); ax.grid(True, axis="x", alpha=0.3)
axes[-1].set_xlabel("share of political-actor coverage")
fig.legend(handles=[Patch(color=POL_COLORS["Government"], label="Government"),
                    Patch(color=POL_COLORS["Opposition"], label="Opposition"),
                    Line2D([0], [0], marker="D", color="w", markerfacecolor="black",
                           markersize=7, label="seat share")],
           loc="upper right", frameon=False)
fig.suptitle("Representation of Parties per Country", y=1.0, fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig5_party_representation.png", bbox_inches="tight")
plt.show()

## 7. Representation per party, per country and topic

Same as figure 6, split into immigration (left) and inflation (right). Party order and seat-share
diamonds are held fixed within each country so the two topics are directly comparable.

In [ ]:
pt = (pol.groupby(["country", "topic_label", "party_final"])
         .agg(actors=("nr_actors", "sum"), gov=("government", "first"),
              seats=("seats", "first")).reset_index())
pt["cov"] = pt.groupby(["country", "topic_label"])["actors"].transform(lambda s: s / s.sum())

counts = party.groupby("country").size().reindex(h.COUNTRY_ORDER)
fig, axes = plt.subplots(len(h.COUNTRY_ORDER), 2, figsize=(14, 0.42 * counts.sum() + 2),
                         gridspec_kw={"height_ratios": counts.values})
for i, cc in enumerate(h.COUNTRY_ORDER):
    order = party[party["country"] == cc].sort_values("cov")
    parties = order["party_final"].tolist()
    govmap  = dict(zip(order["party_final"], order["gov"]))
    seatmap = dict(zip(order["party_final"], order["seats"]))
    y = np.arange(len(parties))
    xmax = max(pt.loc[pt["country"] == cc, "cov"].max(), order["seats"].max()) * 1.08
    for j, tp in enumerate(["immigration", "inflation"]):
        ax = axes[i, j]
        sub = pt[(pt["country"] == cc) & (pt["topic_label"] == tp)].set_index("party_final")
        cov = np.array([sub["cov"].get(p, 0.0) for p in parties])
        colors = [POL_COLORS["Government"] if govmap[p] == 1 else POL_COLORS["Opposition"] for p in parties]
        ax.barh(y, cov, color=colors, height=0.7, zorder=2)
        ax.scatter([seatmap[p] for p in parties], y, color="black", marker="D", s=16, zorder=3)
        ax.set_yticks(y)
        ax.set_yticklabels(parties if j == 0 else [], fontsize=7)
        ax.set_xlim(0, xmax); ax.grid(True, axis="x", alpha=0.3)
        if i == 0: ax.set_title(TOPIC_DISPLAY[tp])
        if j == 0: ax.set_ylabel(h.COUNTRY_NAME[cc], fontsize=10)
for ax in axes[-1]:
    ax.set_xlabel("share of political-actor coverage")
fig.legend(handles=[Patch(color=POL_COLORS["Government"], label="Government"),
                    Patch(color=POL_COLORS["Opposition"], label="Opposition"),
                    Line2D([0], [0], marker="D", color="w", markerfacecolor="black",
                           markersize=7, label="seat share")],
           loc="upper right", frameon=False)
fig.suptitle("Representation of Parties per Country and Topic", y=1.0, fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig6_party_representation_topic.png", bbox_inches="tight")
plt.show()
print("saved 7 figures to", FIG_DIR)